# LLM Translation Metrics Aggregation

This notebook aggregates translation metrics (BLEU, chrF, TER, BERT) from LLM experiments and generates comparison visualizations.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
# Configuration
CLEANED_DATA_DIR = Path('cleaned_data')
OUTPUT_DIR = Path('aggregated_metrics')
CHARTS_DIR = OUTPUT_DIR / 'comparison_charts'

# Metrics to aggregate
METRICS = ['bleu_score', 'chrF_score', 'ter_test', 'bert_score_f1']

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
CHARTS_DIR.mkdir(exist_ok=True)

## 1. Load and Aggregate Data

In [3]:
def load_json_file(filepath):
    """Load a JSON file and return its contents."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

def compute_averages(data):
    """Compute average metrics from a list of prompt results."""
    if not data:
        return None
    
    averages = {}
    for metric in METRICS:
        values = [item.get(metric, 0) for item in data if metric in item]
        if values:
            averages[f'avg_{metric}'] = sum(values) / len(values)
        else:
            averages[f'avg_{metric}'] = 0
    
    averages['sample_count'] = len(data)
    return averages

def get_round_number(filename):
    """Extract round number from filename (e.g., '1.json' -> 1)."""
    return int(Path(filename).stem)

In [4]:
def process_all_data():
    """Process all LLM data and return aggregated results."""
    all_results = []
    
    # Get all LLM folders
    llm_folders = [d for d in CLEANED_DATA_DIR.iterdir() if d.is_dir()]
    
    for llm_folder in sorted(llm_folders):
        llm_name = llm_folder.name
        print(f"Processing {llm_name}...")
        
        # Walk through all JSON files
        for json_file in llm_folder.rglob('*.json'):
            # Get relative path for category info
            rel_path = json_file.relative_to(llm_folder)
            category = str(rel_path.parent)
            round_num = get_round_number(json_file.name)
            
            # Load and compute averages
            data = load_json_file(json_file)
            averages = compute_averages(data)
            
            if averages:
                result = {
                    'llm': llm_name,
                    'category': category,
                    'round': round_num,
                    **averages
                }
                all_results.append(result)
                
                # Save individual JSON file
                output_path = OUTPUT_DIR / llm_name / category
                output_path.mkdir(parents=True, exist_ok=True)
                
                output_file = output_path / f'round_{round_num}_avg.json'
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump(result, f, indent=2)
    
    return all_results

# Process all data
all_results = process_all_data()
print(f"\nTotal aggregated results: {len(all_results)}")

Processing claude3.5...
Processing gemini2.5...
Processing gpt5...
Processing llama3_1b...
Processing llama3_8b...
Processing mistral...


Processing phi3_14b...
Processing phi3_8b...
Processing qwen_14b...

Total aggregated results: 153


## 2. Create Summary DataFrame

In [5]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(all_results)
df.head(10)

,llm,category,round,avg_bleu_score,avg_chrF_score,avg_ter_test,avg_bert_score_f1,sample_count
0,claude3.5,grammatical_induction,1,54.284941,66.221288,73.428571,0.944407,5
1,claude3.5,grammatical_induction,2,22.707722,51.472255,57.777778,0.907586,5
2,claude3.5,grammatical_induction,3,28.994246,45.267076,60.111111,0.901950,5
3,claude3.5,grammatical_induction,4,35.278291,56.182666,58.686869,0.921532,5
4,claude3.5,grammatical_induction,5,14.389460,45.614222,100.181993,0.912811,5
5,claude3.5,morphological_induction,1,57.512442,74.502619,79.152237,0.935140,15
6,claude3.5,zero_shot,1,3.603309,9.903597,203.333333,0.866242,5
7,claude3.5,few_shot\translation_question,1,25.645595,29.837534,100.000000,0.900620,5
8,claude3.5,few_shot\translation_question,2,26.648216,37.530204,56.095238,0.894184,5
9,claude3.5,few_shot\translation_question,3,25.595331,50.541192,71.095238,0.919890,5


In [6]:
# Aggregate by LLM and round (across all categories) using weighted averages
def wavg(group, avg_name, weight_name):
    d = group[avg_name]
    w = group[weight_name]
    return (d * w).sum() / w.sum() if w.sum() > 0 else 0

summary_df = df.groupby(['llm', 'round']).apply(lambda g: pd.Series({
    'avg_bleu_score': wavg(g, 'avg_bleu_score', 'sample_count'),
    'avg_chrF_score': wavg(g, 'avg_chrF_score', 'sample_count'),
    'avg_ter_test': wavg(g, 'avg_ter_test', 'sample_count'),
    'avg_bert_score_f1': wavg(g, 'avg_bert_score_f1', 'sample_count'),
    'sample_count': g['sample_count'].sum()
})).reset_index()

summary_df

,llm,round,avg_bleu_score,avg_chrF_score,avg_ter_test,avg_bert_score_f1,sample_count
0,claude3.5,1,35.912316,53.401384,135.110660,0.907956,40.0
1,claude3.5,2,42.250321,58.427481,63.801587,0.919118,20.0
2,claude3.5,3,47.182928,60.776808,56.134921,0.929788,20.0
3,claude3.5,4,34.238281,56.738446,71.505051,0.929787,20.0
4,claude3.5,5,41.573829,59.763957,81.295498,0.932639,20.0
5,gemini2.5,1,57.188828,64.954873,47.164683,0.939232,40.0
6,gemini2.5,2,42.797260,57.296585,50.829365,0.916251,20.0
7,gemini2.5,3,42.017381,58.405825,83.599206,0.918280,20.0
8,gemini2.5,4,37.159569,50.505864,70.560606,0.838384,20.0
9,gemini2.5,5,44.697940,57.640005,76.082803,0.921279,20.0


### Composite Score
Composite Score = BLEU + chrF + (BERTScore_F1 × 100) − TER
Note: Higher is better. TER is subtracted because lower TER indicates better performance.

In [7]:
# Compute composite score uniformly for all models
summary_df['composite_score'] = (
    summary_df['avg_bleu_score'] + 
    summary_df['avg_chrF_score'] + 
    (summary_df['avg_bert_score_f1'] * 100) - 
    summary_df['avg_ter_test']
)
summary_df

,llm,round,avg_bleu_score,avg_chrF_score,avg_ter_test,avg_bert_score_f1,sample_count,composite_score
0,claude3.5,1,35.912316,53.401384,135.110660,0.907956,40.0,44.998627
1,claude3.5,2,42.250321,58.427481,63.801587,0.919118,20.0,128.788026
2,claude3.5,3,47.182928,60.776808,56.134921,0.929788,20.0,144.803583
3,claude3.5,4,34.238281,56.738446,71.505051,0.929787,20.0,112.450328
4,claude3.5,5,41.573829,59.763957,81.295498,0.932639,20.0,113.306160
5,gemini2.5,1,57.188828,64.954873,47.164683,0.939232,40.0,168.902250
6,gemini2.5,2,42.797260,57.296585,50.829365,0.916251,20.0,140.889551
7,gemini2.5,3,42.017381,58.405825,83.599206,0.918280,20.0,108.651975
8,gemini2.5,4,37.159569,50.505864,70.560606,0.838384,20.0,100.943267
9,gemini2.5,5,44.697940,57.640005,76.082803,0.921279,20.0,118.383039


In [8]:
# Task 3: Compute Standard Deviations of round-level averages
# The unit of variance is round-level averages. We group by LLM and take the std dev.
std_df = summary_df.groupby('llm').agg({
    'avg_bleu_score': ['mean', 'std'],
    'avg_chrF_score': ['mean', 'std'],
    'avg_ter_test': ['mean', 'std'],
    'avg_bert_score_f1': ['mean', 'std']
})

# Flatten MultiIndex columns
std_df.columns = [f"{metric}_{stat}" for metric, stat in std_df.columns]
std_df = std_df.reset_index()

# Format to required table layout
table1 = pd.DataFrame({
    'Model': std_df['llm'],
    'Avg BLEU': std_df['avg_bleu_score_mean'].round(2),
    'Std BLEU': std_df['avg_bleu_score_std'].round(2),
    'Avg chrF': std_df['avg_chrF_score_mean'].round(2),
    'Std chrF': std_df['avg_chrF_score_std'].round(2),
    'Avg TER': std_df['avg_ter_test_mean'].round(2),
    'Std TER': std_df['avg_ter_test_std'].round(2),
    'Avg BERTScore': std_df['avg_bert_score_f1_mean'].round(2),
    'Std BERTScore': std_df['avg_bert_score_f1_std'].round(2)
})

# Save and print
table1.to_csv(OUTPUT_DIR / 'table1_with_stddev.csv', index=False)
print(table1.to_string(index=False))

    Model  Avg BLEU  Std BLEU  Avg chrF  Std chrF  Avg TER  Std TER  Avg BERTScore  Std BERTScore
claude3.5     40.23      5.21     57.82      2.90    81.57    31.35           0.92           0.01
gemini2.5     44.77      7.48     57.76      5.12    65.65    15.94           0.91           0.04
     gpt5     33.44      8.52     49.57      7.08    96.56     9.44           0.91           0.01
llama3_1b      5.22      3.32     13.71      3.33   131.04    13.14           0.78           0.14
llama3_8b     17.31      6.81     40.86      4.97   105.53    22.45           0.83           0.15
  mistral     21.33      6.89     44.44      8.35   185.82    23.45           0.81           0.15
 phi3_14b     20.01      6.96     42.32      7.16   185.87    55.33           0.82           0.15
  phi3_8b     10.62      3.73     34.54      5.24   511.92   242.40           0.81           0.15
 qwen_14b     31.83      6.63     53.49      5.94    90.88    22.29           0.85           0.16


In [9]:
# Save summary to CSV
summary_csv_path = CHARTS_DIR / 'all_models_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f"Saved summary to {summary_csv_path}")

Saved summary to aggregated_metrics\comparison_charts\all_models_summary.csv


## 3. Generate Refined Comparison Charts

In [10]:
# Set style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})

def generate_refined_charts(df):
    # Metrics mapping
    metrics_map = {
        'avg_bleu_score': 'BLEU Score',
        'avg_chrF_score': 'chrF Score',
        'avg_ter_test': 'TER Score',
        'avg_bert_score_f1': 'BERTScore F1'
    }
    
    unique_rounds = sorted(df['round'].unique())
    unique_llms = sorted(df['llm'].unique())
    
    print(f"Generating per-round charts for {len(unique_rounds)} rounds...")
    
    # 1. Per-Round, Per-Metric Charts
    for r in unique_rounds:
        round_df = df[df['round'] == r]
        
        for metric_col, metric_name in metrics_map.items():
            plt.figure(figsize=(10, 6))
            
            # Create bar plot
            ax = sns.barplot(
                data=round_df,
                x='llm',
                y=metric_col,
                order=unique_llms,
                palette='viridis',
                hue='llm',
                legend=False
            )
            
            # Customization
            plt.title(f'Round {r} - {metric_name} Comparison')
            plt.xlabel('LLM Model')
            plt.ylabel(metric_name)
            plt.xticks(rotation=45)
            
            # Add value labels
            for container in ax.containers:
                ax.bar_label(container, fmt='%.2f', padding=3)
            
            plt.tight_layout()
            
            # Save file
            # Clean up filename for bert
            safe_metric_name = metric_col.replace('avg_', '').replace('_score', '').replace('_test', '')
            if 'bert' in safe_metric_name:
                safe_metric_name = safe_metric_name.replace('_f1', '')
                
            filename = f"round_{r}_{safe_metric_name}.png"
            output_path = CHARTS_DIR / filename
            plt.savefig(output_path)
            plt.close()
            print(f"  Saved: {output_path}")

    # 2. Overall Summary (Average of Round Averages)
    print("\nGenerating overall summary (Average of Round Averages)...")
    
    # Group by LLM and calculate mean of the round-level averages
    # We use the existing summary_df which is already averaged by round
    overall_avg_df = df.groupby('llm').apply(lambda g: pd.Series({
        m: (g[m] * g['sample_count']).sum() / g['sample_count'].sum() if g['sample_count'].sum() > 0 else 0
        for m in metrics_map.keys()
    })).reset_index()
    
    # Save this summary to CSV
    overall_csv_path = CHARTS_DIR / 'overall_average_of_rounds.csv'
    overall_avg_df.to_csv(overall_csv_path, index=False)
    print(f"  Saved summary data: {overall_csv_path}")
    
    # Generate charts for overall averages
    for metric_col, metric_name in metrics_map.items():
        plt.figure(figsize=(10, 6))
        
        ax = sns.barplot(
            data=overall_avg_df,
            x='llm',
            y=metric_col,
            order=unique_llms,
            palette='magma',
            hue='llm',
            legend=False
        )
        
        plt.title(f'Overall Average (Across Rounds) - {metric_name}')
        plt.xlabel('LLM Model')
        plt.ylabel(f'Average {metric_name}')
        plt.xticks(rotation=45)
        
        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', padding=3)
            
        plt.tight_layout()
        
        safe_metric_name = metric_col.replace('avg_', '').replace('_score', '').replace('_test', '')
        if 'bert' in safe_metric_name:
            safe_metric_name = safe_metric_name.replace('_f1', '')
            
        filename = f"overall_average_{safe_metric_name}.png"
        output_path = CHARTS_DIR / filename
        plt.savefig(output_path)
        plt.close()
        print(f"  Saved: {output_path}")

# Run the generation
generate_refined_charts(summary_df)

Generating per-round charts for 5 rounds...


  Saved: aggregated_metrics\comparison_charts\round_1_bleu.png
  Saved: aggregated_metrics\comparison_charts\round_1_chrF.png


  Saved: aggregated_metrics\comparison_charts\round_1_ter.png
  Saved: aggregated_metrics\comparison_charts\round_1_bert.png


  Saved: aggregated_metrics\comparison_charts\round_2_bleu.png
  Saved: aggregated_metrics\comparison_charts\round_2_chrF.png


  Saved: aggregated_metrics\comparison_charts\round_2_ter.png
  Saved: aggregated_metrics\comparison_charts\round_2_bert.png


...

  Saved: aggregated_metrics\comparison_charts\round_4_ter.png
  Saved: aggregated_metrics\comparison_charts\round_4_bert.png


  Saved: aggregated_metrics\comparison_charts\round_5_bleu.png
  Saved: aggregated_metrics\comparison_charts\round_5_chrF.png


  Saved: aggregated_metrics\comparison_charts\round_5_ter.png
  Saved: aggregated_metrics\comparison_charts\round_5_bert.png

Generating overall summary (Average of Round Averages)...
  Saved summary data: aggregated_metrics\comparison_charts\overall_average_of_rounds.csv


  Saved: aggregated_metrics\comparison_charts\overall_average_bleu.png
  Saved: aggregated_metrics\comparison_charts\overall_average_chrF.png


  Saved: aggregated_metrics\comparison_charts\overall_average_ter.png
  Saved: aggregated_metrics\comparison_charts\overall_average_bert.png


In [11]:
# Task 5: Generate Box-and-Whisker Plots
import seaborn as sns
import matplotlib.pyplot as plt

# X-axis short model names mapping
model_names_map = {
    'claude3.5': 'Claude 3.5',
    'gemini2.5': 'Gemini 2.5',
    'gpt5': 'GPT-5',
    'llama3_1b': 'Llama 1B',
    'llama3_8b': 'Llama 8B',
    'mistral': 'Mistral',
    'phi3_14b': 'Phi3 14B',
    'phi3_8b': 'Phi3 8B',
    'qwen_14b': 'Qwen 14B'
}

# Apply short names
plot_df = summary_df.copy()
plot_df['llm_short'] = plot_df['llm'].map(model_names_map)

# Order by descending average BLEU (calculate from summary_df)
order_df = plot_df.groupby('llm_short')['avg_bleu_score'].mean().sort_values(ascending=False)
model_order = order_df.index.tolist()

# Mean of frontier models
frontier_models = ['Claude 3.5', 'Gemini 2.5', 'GPT-5']
frontier_mean_bleu = plot_df[plot_df['llm_short'].isin(frontier_models)]['avg_bleu_score'].mean()
frontier_mean_bert = plot_df[plot_df['llm_short'].isin(frontier_models)]['avg_bert_score_f1'].mean()

# Plot BLEU
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")
ax = sns.boxplot(x='llm_short', y='avg_bleu_score', data=plot_df, order=model_order, color='lightgray', showfliers=False)
sns.stripplot(x='llm_short', y='avg_bleu_score', data=plot_df, order=model_order, color='black', alpha=0.6, jitter=True)

plt.axhline(frontier_mean_bleu, color='red', linestyle='--', label=f'Frontier Mean: {frontier_mean_bleu:.2f}')
plt.title("BLEU Score Distribution Across Prompt Complexity Levels (Rounds 1–5)")
plt.xlabel("Model")
plt.ylabel("BLEU Score")
plt.legend()
plt.tight_layout()

Path('figures').mkdir(exist_ok=True)
plt.savefig('figures/boxplot_bleu_by_model.png', dpi=300)
plt.close()

# Plot BERTScore F1
plt.figure(figsize=(12, 6))
ax = sns.boxplot(x='llm_short', y='avg_bert_score_f1', data=plot_df, order=model_order, color='lightgray', showfliers=False)
sns.stripplot(x='llm_short', y='avg_bert_score_f1', data=plot_df, order=model_order, color='black', alpha=0.6, jitter=True)

plt.axhline(frontier_mean_bert, color='red', linestyle='--', label=f'Frontier Mean: {frontier_mean_bert:.3f}')
plt.title("BERTScore F1 Distribution Across Prompt Complexity Levels (Rounds 1–5)")
plt.xlabel("Model")
plt.ylabel("BERTScore F1")
plt.legend()
plt.tight_layout()

plt.savefig('figures/boxplot_bertscore_by_model.png', dpi=300)
plt.close()
print("Box plots generated in figures/ directory.")


Box plots generated in figures/ directory.
